# PERSISTANCE IN LANGGRAPH

In [16]:
# libraries
import os
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq.chat_models import ChatGroq
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import SystemMessage, HumanMessage
from typing import TypedDict
from dotenv import load_dotenv

In [8]:
# load api key
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [4]:
# --- define state ---
class State(TypedDict):
    topic: str
    joke: str
    explanation: str

In [9]:
# --- Define LLM Model ---
MODEL = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature = 0.6,
    max_tokens = None
)

In [11]:
# --- Define functions ---
def generate_joke(state: State):
    messages = [
        SystemMessage(content = "You are an expert in generating jokes"),
        HumanMessage(content = f"Generate a joke on the {state['topic']}.")
    ]
    response = MODEL.invoke(messages).content
    return {"joke": response}

In [12]:
# --- Define functions ---
def explain_joke(state: State):
    messages = [
        SystemMessage(content = "You are an expert in explaining jokes"),
        HumanMessage(content = f"Explain the joke:\n{state['joke']}")
    ]
    response = MODEL.invoke(messages).content
    return {"explanation": response}

In [20]:
# --- Build graph ---
graph = StateGraph(State)
graph.add_node("generate_joke", generate_joke)
graph.add_node("explain_joke", explain_joke)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "explain_joke")
graph.add_edge("explain_joke", END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer)

In [25]:
# --- Invoke graph ---
config: RunnableConfig = {"configurable": {"thread_id": "1"}}
initial_state = {
    "topic": "Cricket",
    "joke": "",
    "explanation": ""
}
print(workflow.invoke(initial_state, config))

{'topic': 'Cricket', 'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a "sticky" wicket situation and was feeling a little "bowled" over! (get it?)', 'explanation': 'A joke that\'s a pitch-perfect blend of wordplay and cricket terminology.\n\nHere\'s a breakdown of the joke:\n\n1. **Setup**: The joke starts with a classic "Why did [animal] go to the doctor?" format, which primes the listener to expect a punchline that\'s a play on words.\n2. **Cricket terminology**: The joke relies on the listener being familiar with cricket terms. In cricket:\n\t* A "wicket" refers to the set of three stumps and two bails that a batsman defends. A "sticky wicket" is a phrase used to describe a difficult or precarious situation, often originating from the idea that a wet or sticky pitch can make it hard for batsmen to play.\n\t* To be "bowled over" means to be knocked over or defeated by a bowler\'s delivery, resulting in the batsman being dismissed.\n3. **Wordplay**: The joke uses word

In [26]:
# fetch latest state
config = {"configurable": {"thread_id": "1"}}
workflow.get_state(config)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a "sticky" wicket situation and was feeling a little "bowled" over! (get it?)', 'explanation': 'A joke that\'s a pitch-perfect blend of wordplay and cricket terminology.\n\nHere\'s a breakdown of the joke:\n\n1. **Setup**: The joke starts with a classic "Why did [animal] go to the doctor?" format, which primes the listener to expect a punchline that\'s a play on words.\n2. **Cricket terminology**: The joke relies on the listener being familiar with cricket terms. In cricket:\n\t* A "wicket" refers to the set of three stumps and two bails that a batsman defends. A "sticky wicket" is a phrase used to describe a difficult or precarious situation, often originating from the idea that a wet or sticky pitch can make it hard for batsmen to play.\n\t* To be "bowled over" means to be knocked over or defeated by a bowler\'s delivery, resulting in the batsman being dismissed.\n3. **Wordplay*

In [28]:
# fetch state hisotory
config = {"configurable": {"thread_id": "1"}}
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket go to the doctor?\n\nBecause it had a "sticky" wicket situation and was feeling a little "bowled" over! (get it?)', 'explanation': 'A joke that\'s a pitch-perfect blend of wordplay and cricket terminology.\n\nHere\'s a breakdown of the joke:\n\n1. **Setup**: The joke starts with a classic "Why did [animal] go to the doctor?" format, which primes the listener to expect a punchline that\'s a play on words.\n2. **Cricket terminology**: The joke relies on the listener being familiar with cricket terms. In cricket:\n\t* A "wicket" refers to the set of three stumps and two bails that a batsman defends. A "sticky wicket" is a phrase used to describe a difficult or precarious situation, often originating from the idea that a wet or sticky pitch can make it hard for batsmen to play.\n\t* To be "bowled over" means to be knocked over or defeated by a bowler\'s delivery, resulting in the batsman being dismissed.\n3. **Wordplay